In [ ]:
import pandas as pd
import numpy as np

import datetime
import os, sys
import importlib

import utils
importlib.reload(utils)

from utils import plot_series, plot_series_with_names, plot_series_bar
from utils import plot_dataframe
from utils import get_universe_adjusted_series, scale_weights_to_one, scale_to_book_long_short
from utils import generate_portfolio, backtest_portfolio
from utils import match_implementations

import plotly.graph_objects as go

import warnings
warnings.filterwarnings("ignore")

In [ ]:
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "stores")

features = pd.read_parquet(os.path.join(DATA_DIR, "features.parquet"))

universe = pd.read_parquet(os.path.join(DATA_DIR, "universe_1m.parquet"))
 
returns = pd.read_parquet(os.path.join(DATA_DIR, "returns.parquet"))

In [ ]:
sector_mapping = pd.read_csv(os.path.join(BASE_DIR, "top_5000_us_by_marketcap.csv")).set_index("symbol").sector

In [150]:
# Testing the backtest_portfolio function for a specific signal

def generate_portfolio_vectorized(
    entire_features: pd.DataFrame,
    universe: pd.DataFrame,
    start_date: str,
    end_date: str,
    signal_column: str,
    neutralize_by: str = None,
    sector_mapping: pd.DataFrame = None
):
    # Validate date format
    try:
        start_dt = datetime.datetime.strptime(start_date, '%Y-%m-%d')
        end_dt = datetime.datetime.strptime(end_date, '%Y-%m-%d')
        cutoff_date = datetime.datetime.strptime('2005-01-01', '%Y-%m-%d')
    except ValueError:
        raise ValueError("start_date and end_date must be strings in 'YYYY-MM-DD' format.")

    # Ensure start_date is before end_date
    if start_dt >= end_dt:
        raise ValueError("start_date must be earlier than end_date.")

    # Ensure start_date is not before '2005-01-01'
    if start_dt < cutoff_date:
        raise ValueError("start_date must be later than '2005-01-01'.")

    # Get trading days within the date range
    trading_days = universe.index[(universe.index >= start_dt) & (universe.index <= end_dt)]

    if len(trading_days) == 0:
        raise ValueError("No Trading Days in the specified dates")
    
    portfolio = 0

    universe_boolean = universe.loc[:end_date].astype(bool)
    
    features_ = entire_features.loc[:end_date]

    signal1 = features_[signal_column].shift(5)
    signal1 = signal1.where(universe_boolean, np.nan)
    signal1 = signal1.rank(axis=1, method="min", ascending=True)
    if neutralize_by == "market":
        signal1 = signal1.sub(signal1.mean(axis=1), axis=0)
        signal1 = signal1.sub(signal1.mean(axis=1), axis=0)
        signal1 = signal1.div(signal1.abs().sum(axis=1), axis=0)
        signal1 = signal1.fillna(0)
    if neutralize_by == "sector" and sector_mapping is not None:
        sector_mapping = sector_mapping.reindex(signal1.columns, fill_value="Others")
        sector_mapping = sector_mapping.loc[signal1.columns]
        signal1 = signal1.sub(signal1.groupby(sector_mapping, axis=1).transform("mean"), axis=0)
        signal1 = signal1.div(signal1.abs().sum(axis=1), axis=0)
        signal1 = signal1.fillna(0)

    portfolio = -1 * signal1

    portfolio = portfolio.div(portfolio.abs().sum(axis=1), axis=0)
    
    return portfolio.fillna(0).loc[start_date:end_date]

In [ ]:
benchmark_portfolio_vectorized = generate_portfolio_vectorized(
    features,
    universe,
    "2010-01-01",
    "2026-01-01",
    "trend_1_3",
    "sector",
    sector_mapping
)

In [196]:
benchmark_portfolio_vectorized

ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,VEEA,VNCE,VNRX,VNTG,WKHS,WYHG,YSXT,YXT,ZDAI,ZKIN
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-06,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,0.000410,-0.000526,-0.000055,0.000881,0.000780,0.000851,0.000487,-0.000072,0.000610,0.000035,...,-0.0,-0.0,-0.001279,-0.0,-0.0,-0.0,0.000825,-0.0,-0.0,-0.0
2025-12-26,-0.000070,0.000569,0.000814,-0.000553,-0.000547,-0.000326,0.000723,-0.001228,-0.000759,-0.000558,...,-0.0,-0.0,0.000551,-0.0,-0.0,-0.0,0.000429,-0.0,-0.0,-0.0
2025-12-29,-0.000504,-0.000786,-0.000123,0.000461,0.000041,-0.000806,-0.000425,0.000341,-0.000840,-0.000640,...,-0.0,-0.0,0.000948,-0.0,-0.0,-0.0,0.000461,-0.0,-0.0,-0.0


In [197]:
sr_vectorized, pnl_vectorized = backtest_portfolio(benchmark_portfolio_vectorized.loc["2010":"2019"], returns.loc["2010":"2019"], universe.loc["2010":"2019"], True, True)

Gross Sharpe Ratio:  -0.244
Net Sharpe Ratio:  -1.617
Turnover %:  149.53


In [ ]:
feature_results = pd.DataFrame(columns=["feature", "sharpe_ratio"])
feature_pnls = {}
for col in features.columns.get_level_values(0).unique():
    print(f"Testing feature: {col}")
    portfolio_vectorized = generate_portfolio_vectorized(
        features,
        universe,
        "2010-01-01",
        "2026-01-01",
        col,
        "sector",
        sector_mapping
    )
    sr_vectorized, pnl_vectorized = backtest_portfolio(portfolio_vectorized.loc["2010":"2019"], returns.loc["2010":"2019"], universe.loc["2010":"2019"], False, False)
    new_row = pd.DataFrame([{
        "feature": col,
        "sharpe_ratio": sr_vectorized
    }])
    feature_pnls[col] = pnl_vectorized

    feature_results = pd.concat([feature_results, new_row], ignore_index=True)

Testing feature: relative_strength_index
Testing feature: williams_r
Testing feature: rsi
Testing feature: volatility_20
Testing feature: volatility_60
Testing feature: trend_1_3
Testing feature: trend_5_20
Testing feature: trend_20_60
Testing feature: average_true_range
Testing feature: macd
Testing feature: macd_signal
Testing feature: macd_histogram
Testing feature: trix
Testing feature: commodity_channel_index
Testing feature: chande_momentum_oscillator
Testing feature: ichimoku_conversion
Testing feature: ichimoku_base
Testing feature: ichimoku_leading_a
Testing feature: ichimoku_leading_b
Testing feature: know_sure_thing
Testing feature: ultimate_oscillator
Testing feature: aroon_up
Testing feature: aroon_down
Testing feature: aroon_oscillator
Testing feature: stochastic_k
Testing feature: stochastic_d
Testing feature: on_balance_volume
Testing feature: ease_of_movement
Testing feature: chaikin_money_flow
Testing feature: accumulation_distribution_index
Testing feature: volume


In [188]:
top_signals = feature_results.sort_values(by="sharpe_ratio", ascending=False)
top_signals

,feature,sharpe_ratio
8,average_true_range,2.298
15,ichimoku_conversion,1.655
16,ichimoku_base,1.641
17,ichimoku_leading_a,1.640
18,ichimoku_leading_b,1.636
29,accumulation_distribution_index,0.606
9,macd,0.528
25,stochastic_d,0.479
10,macd_signal,0.453
13,commodity_channel_index,0.414


In [204]:
selected_signals = pd.DataFrame(feature_pnls).loc[:, ['ichimoku_conversion', 'accumulation_distribution_index', 'macd']]

In [205]:
selected_signals.corr()

,ichimoku_conversion,accumulation_distribution_index,macd
ichimoku_conversion,1.000000,0.857898,0.410614
accumulation_distribution_index,0.857898,1.000000,0.413869
macd,0.410614,0.413869,1.000000


In [ ]:
selected_signals.to_parquet(os.path.join(DATA_DIR, "selected_signals.parquet"))